# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [13]:
import pandas as pd
PATH = '../06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()




,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [17]:
# TODO (hint: .str.lower().str.contains('pop'))


!git clone https://github.com/jasontpv/DS2002FA26.git
cat = pd.read_csv('DS2002FA26/notebooks/06-midterm-walmart/data/product_catalog.csv')

x = cat['product_name'].str.lower().str.contains('pop', na=False)
pop = cat[x]
print(f"Rows that look like Pop-Tart products: {len(pop)}")
pop

fatal: destination path 'DS2002FA26' already exists and is not an empty directory.
Rows that look like Pop-Tart products: 4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [21]:
# TODO
name = cat['product_name'].str.lower()

# All Pop-Tart rows (tighter than just 'pop')
poptart = cat[name.str.contains(r'pop.?tart', na=False)]

# Split into strawberry vs. everything else
is_straw = poptart['product_name'].str.lower().str.contains('strawberry', na=False)
straw = poptart[is_straw]
other = poptart[~is_straw]

print(f"Strawberry Pop-Tart rows: {len(straw)}")
print(straw[['sku', 'product_name']].to_string(index=False))

print(f"\nPop-Tart rows that are NOT strawberry ({len(other)}) - keep separate:")
print(other[['sku', 'product_name', 'category']].to_string(index=False))


Strawberry Pop-Tart rows: 4
          sku         product_name
        PT-12 pop-tarts strawberry
   SKU#459812  Strawberry Pop-Tart
POPTART-STRAW Pop-Tarts Strawberry
PT_STRAWBERRY pop-tarts strawberry

Pop-Tart rows that are NOT strawberry (0) - keep separate:
Empty DataFrame
Columns: [sku, product_name, category]
Index: []


**Pop-Tart rows that are a different product:** _..._ None are present. All 4 Pop-Tart rows in `product_catalog.csv` (`PT-12`, `SKU#459812`, `POPTART-STRAW`, `PT_STRAWBERRY`) are strawberry, as the empty `other_pop` output above shows. If any other flavor appeared, it would stay separate from the strawberry group and not be merged into it.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [22]:
# TODO

# Canonical SKU for strawberry Pop-Tarts (any of the four works; PT-12 is the shortest)
CANON = 'PT-12'

sku_map = {
    'PT-12':         CANON,
    'SKU#459812':    CANON,
    'POPTART-STRAW': CANON,
    'PT_STRAWBERRY': CANON,
}

# Safe: unmapped SKUs stay as themselves
cat['canonical_sku'] = cat['sku'].map(sku_map).fillna(cat['sku'])

# Equivalent one-liner:
# cat['canonical_sku'] = cat['sku'].replace(sku_map)

# Sanity checks
print(cat['canonical_sku'].isna().sum(), "NaNs (should be 0)")
print(len(cat), "rows before,", len(cat), "rows after")
print(cat[['sku', 'canonical_sku']].head(6))
print(cat['canonical_sku'].nunique(), "unique canonical SKUs vs", cat['sku'].nunique(), "unique original SKUs")

0 NaNs (should be 0)
45 rows before, 45 rows after
             sku canonical_sku
0          PT-12         PT-12
1     SKU#459812         PT-12
2  POPTART-STRAW         PT-12
3  PT_STRAWBERRY         PT-12
4         WTR-24        WTR-24
5      WATER24PK     WATER24PK
42 unique canonical SKUs vs 45 unique original SKUs


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [23]:
# TODO

# Before
print("BEFORE:")
print(cat['category'].value_counts(dropna=False).sort_index())
print(cat['category'].nunique(dropna=False), "distinct values\n")

# Normalize: strip whitespace, then title-case
cat['category_clean'] = cat['category'].str.strip().str.title()

# After
print("AFTER:")
print(cat['category_clean'].value_counts(dropna=False).sort_index())
print(cat['category_clean'].nunique(dropna=False), "distinct values")

BEFORE:
category
ALCOHOL                1
Alcoholic Beverages    1
Baby                   2
Baby Products          1
Breakfast              2
Breakfast Foods        1
ELECTRONICS            2
EMERGENCY              3
ENTERTAINMENT          3
Electronics            1
Emergency              5
Emergency Supplies     3
GROCERY                3
Groceries              1
Grocery                1
HOUSEHOLD              1
Household              1
Snacks                 1
baby                   1
breakfast              1
electronics            1
emergency              1
entertainment          2
grocery                1
health                 2
household              2
snacks                 1
Name: count, dtype: int64
27 distinct values

AFTER:
category_clean
Alcohol                1
Alcoholic Beverages    1
Baby                   3
Baby Products          1
Breakfast              3
Breakfast Foods        1
Electronics            4
Emergency              9
Emergency Supplies     3
Entertainment  

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [25]:
# TODO

import re

def norm_name(s):
    if pd.isna(s):
        return None
    s = s.lower()
    s = re.sub(r'[^a-z0-9\s]', ' ', s)      # strip punctuation (hyphens become spaces)
    return ' '.join(sorted(s.split()))       # sort the words

cat['name_key'] = cat['product_name'].map(norm_name)

# Groups with more than one distinct SKU
groups = (cat.dropna(subset=['name_key'])
             .groupby('name_key')['sku']
             .agg(['nunique', list])
             .query('nunique > 1')
             .sort_values('nunique', ascending=False))

print(len(groups), "candidate groups")
groups

dupe_keys = groups.index
cat[cat['name_key'].isin(dupe_keys)].sort_values('name_key')[
    ['name_key', 'sku', 'product_name', 'category', 'package_size', 'supplier']
]

14 candidate groups


,name_key,sku,product_name,category,package_size,supplier
12,24 beer light pack,BEER-24,Light Beer (24-pack),Alcoholic Beverages,bundle,Hasbro
13,24 beer light pack,LTBEER24,Light Beer (24-pack),ALCOHOL,bundle,Kellogg's
8,4 aa batteries pack,BAT-AA4,AA Batteries 4-Pack,Emergency Supplies,4pk,Kimberly-Clark
9,4 aa batteries pack,BATT_AA_4PK,AA Batteries 4-Pack,Emergency,24 pack,Anheuser-Busch
30,aid first kit,FIRSTAID-KIT,First Aid Kit,health,4 pack,J.M. Smucker
29,aid first kit,FAID-01,First-Aid Kit,health,4 pack,Anheuser-Busch
15,board game monopoly,MONOPOLY-01,Monopoly Board Game,entertainment,24-pack,Georgia-Pacific
14,board game monopoly,BG-MONO,Monopoly Board Game,ENTERTAINMENT,1pk,Kimberly-Clark
16,board game monopoly,GAME-MONO,Board Game - Monopoly,ENTERTAINMENT,1pk,Campbell's
24,bread white,WHITEBREAD,WHITE BREAD,grocery,24 pack,Sara Lee


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [ ]:
# TODO

### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

_your answer here_